
# Roofline Model 与 Dtype：为什么 FP16 不一定更 compute-bound？

## 核心问题

传统 roofline model 用：

\[
AI_{std} = \frac{\text{abstract FLOPs}}{\text{HBM bytes}}
\]

认为 FP32 → FP16 时，bytes 减半，FLOPs 不变，所以 $AI_{std}$ 翻倍，workload 更偏向 compute-bound。

**但这个结论可能是错的**，因为：

1. **FLOP 不是电路工作的均匀单位**：FP16 FMA 的晶体管开关次数、能耗、延迟都显著低于 FP32 FMA。
2. **Peak throughput 增长可能比内存带宽快得多**：Ampere/Hopper 的 FP16 Tensor Core peak 可以是 FP32 的 8x~16x。
3. 因此即使 $AI_{std}$ 翻倍，workload 距离该 dtype 的 ridge point 可能反而更远了。

## 本实验目标

同时测量三套量：

| 量 | 定义 | 意义 |
|---|---|---|
| $AI_{std}$ | FLOPs / HBM bytes | 传统 roofline 强度 |
| $I_{bit}$ | bit-level compute proxy / HBM bits | 按位/电路代价归一化 |
| $\rho_{roofline}$ | $AI_{std} / (P_{peak,dtype} / B_{mem})$ | workload 到该 dtype ridge point 的距离 |

$\rho$ 越小，workload 越靠近 memory-bound；$\rho$ 越大，越靠近 compute-bound。

## 实验设置

- GPU: RTX 5060 (Blackwell, sm_120)
- Workloads: GEMM, MLP forward/backward
- Dtypes: FP32, TF32, FP16, BF16
- 测量工具: `torch.profiler` + `nvtx` 标注，必要时用 `ncu` 获取更精确内存指标



## Part 1: 环境设置与工具函数

我们先定义测量函数、FLOPs 估算函数、peak throughput 测量函数。


In [ ]:

import time
import warnings
from collections import defaultdict
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import nvtx

warnings.filterwarnings("ignore")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA: {torch.version.cuda}")
    print(f"PyTorch: {torch.__version__}")


In [ ]:

def benchmark(fn, num_warmups=3, num_trials=5):
    # benchmark function
    for _ in range(num_warmups):
        fn()
    if device == "cuda":
        torch.cuda.synchronize()
    times = []
    for _ in range(num_trials):
        start = time.time()
        fn()
        if device == "cuda":
            torch.cuda.synchronize()
        times.append((time.time() - start) * 1000)
    return sum(times) / len(times), times


def reset_cuda():
    if device == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def bytes_to_str(b):
    for unit in ["B", "KB", "MB", "GB"]:
        if b < 1024:
            return f"{b:.2f} {unit}"
        b /= 1024
    return f"{b:.2f} TB"



## Part 2: 测量各 dtype 的 Peak Compute Throughput

我们用一个大尺寸 GEMM 来逼近各 dtype 的峰值吞吐。

注意：
- FP32: 用普通 `torch.matmul`
- TF32: 设置 `torch.backends.cudnn.allow_tf32 = True`
- FP16/BF16: 用 `torch.matmul`
- 大矩阵有利于让 GPU 进入 compute-bound 状态


In [ ]:

def measure_peak_matmul(dtype: torch.dtype, m: int = 8192, k: int = 8192, n: int = 8192) -> Tuple[float, float]:
    # measure peak matmul
    reset_cuda()

    # TF32 控制
    if dtype == torch.float32:
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
    elif dtype == torch.float32 and "tf32" in str(dtype):
        # 我们用字符串标记区分 TF32 和 FP32
        pass

    a = torch.randn(m, k, device=device, dtype=dtype)
    b = torch.randn(k, n, device=device, dtype=dtype)

    def fn():
        c = torch.matmul(a, b)
        return c

    elapsed_ms, _ = benchmark(fn, num_warmups=5, num_trials=10)

    # FLOPs = 2 * M * K * N
    flops = 2 * m * k * n
    tflops = flops / (elapsed_ms / 1000) / 1e12
    return tflops, elapsed_ms


# 由于 torch dtype 不能区分 FP32 和 TF32，我们用一个辅助结构
DTYPE_CONFIGS = [
    ("FP32", torch.float32, lambda: torch.backends.cuda.matmul.allow_tf32.__set__),
    ("TF32", torch.float32, None),
    ("FP16", torch.float16, None),
    ("BF16", torch.bfloat16, None),
]


def measure_peak_all():
    results = {}
    for name, dtype, _ in DTYPE_CONFIGS:
        if name == "FP32":
            torch.backends.cuda.matmul.allow_tf32 = False
            torch.backends.cudnn.allow_tf32 = False
        elif name == "TF32":
            torch.backends.cuda.matmul.allow_tf32 = True
            torch.backends.cudnn.allow_tf32 = True

        try:
            tflops, elapsed = measure_peak_matmul(dtype)
            results[name] = {"tflops": tflops, "elapsed_ms": elapsed}
            print(f"{name}: {tflops:.2f} TFLOP/s ({elapsed:.2f} ms)")
        except Exception as e:
            print(f"{name} failed: {e}")
            results[name] = {"tflops": 0, "elapsed_ms": 0}
    return results


peak_results = measure_peak_all()



## Part 3: 定义 Workloads

我们测两个典型 workload：

1. **GEMM**: $C = A \times B$，shape $(M, K) \times (K, N)$
2. **MLP**: 多层 Linear + GeLU，forward + backward

对每个 workload，在不同 dtype 下测量：
- 耗时
- HBM bytes（通过 profiler 估计）
- FLOPs


In [ ]:

def gemm_workload(dtype: torch.dtype, m: int = 2048, k: int = 2048, n: int = 2048):
    a = torch.randn(m, k, device=device, dtype=dtype)
    b = torch.randn(k, n, device=device, dtype=dtype)

    def fn():
        c = torch.matmul(a, b)
        return c

    flops = 2 * m * k * n
    return fn, flops


class MLP(nn.Module):
    def __init__(self, dim: int, num_layers: int):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(dim, dim) for _ in range(num_layers)])

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
            x = torch.nn.functional.gelu(x)
        return x


def mlp_workload(dtype: torch.dtype, dim: int = 1024, num_layers: int = 4, batch_size: int = 256):
    model = MLP(dim, num_layers).to(device).to(dtype)
    x = torch.randn(batch_size, dim, device=device, dtype=dtype, requires_grad=True)

    def fn():
        y = model(x).mean()
        y.backward()
        return y

    # 估算 FLOPs：每层 Linear forward 2*batch*dim^2，backward 约 4*batch*dim^2，GeLU 约 8*batch*dim
    per_layer_flops = 2 * batch_size * dim * dim  # forward
    per_layer_flops += 4 * batch_size * dim * dim  # backward
    per_layer_flops += 8 * batch_size * dim        # GeLU approx
    flops = per_layer_flops * num_layers
    return fn, flops



## Part 4: 用 torch.profiler 测量 HBM Bytes

`torch.profiler` 可以给出 CUDA memory 使用情况。我们用 `record_shapes=True` 和 `profile_memory=True` 来追踪。

注意：这个 bytes 是 allocator 层面的统计，不是精确的 HBM traffic。更精确需要用 `ncu`，但 `torch.profiler` 足够做趋势分析。


In [ ]:

from torch.profiler import ProfilerActivity, profile


def profile_workload(name: str, fn, dtype_name: str):
    # 测量 workload，返回耗时、FLOPs、HBM bytes（估算）
    # 设置 TF32
    if dtype_name == "FP32":
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
    elif dtype_name == "TF32":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

    reset_cuda()

    # warmup
    for _ in range(3):
        fn()
    if device == "cuda":
        torch.cuda.synchronize()

    with profile(
        activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
        profile_memory=True,
        record_shapes=True,
        with_stack=False,
    ) as prof:
        fn()
        if device == "cuda":
            torch.cuda.synchronize()

    # 从 profiler 提取 CUDA 总时间
    events = prof.key_averages()
    cuda_time_us = sum(e.device_time_total for e in events if e.device_time_total > 0)

    # 估算 HBM bytes：用 allocated_bytes 峰值 - 初始值（近似）
    mem_stats = torch.cuda.memory_stats() if device == "cuda" else {}
    hbm_bytes = mem_stats.get("active_bytes.all.peak", 0) + mem_stats.get("allocated_bytes.all.peak", 0)

    # 用 nvtx 再跑 benchmark 测 wall-clock
    elapsed_ms, _ = benchmark(fn, num_warmups=2, num_trials=5)

    return {
        "elapsed_ms": elapsed_ms,
        "cuda_time_us": cuda_time_us,
        "hbm_bytes": hbm_bytes,
    }



## Part 5: 运行实验

对 GEMM 和 MLP，分别在 FP32/TF32/FP16/BF16 下测量：

- $T$（耗时）
- $F$（FLOPs）
- $B$（HBM bytes，近似）
- $P_{peak}$（该 dtype 的峰值 TFLOP/s）

然后计算：

\[
AI_{std} = \frac{F}{B}
\]

\[
I_{bit} = \frac{F \times w_{dtype}}{B \times b_{dtype}}
\]

其中 $w_{dtype}$ 是 circuit-level weight（FP32=1.0, TF32≈0.6, FP16≈0.25, BF16≈0.25），$b_{dtype}$ 是每个 element 的 bit 数。

\[
\rho_{roofline} = \frac{AI_{std}}{P_{peak} / B_{mem}}
\]

$B_{mem}$ 是实测或标称的 HBM 带宽。RTX 5060 Laptop 的 HBM（GDDR6/6X）带宽约 128~256 GB/s，这里先用一个保守估计 128 GB/s，并说明这是 scaling 分析，不是绝对值。


In [ ]:

# 定义 dtype 的 bit-level circuit weight
# 这些是经验估计，用于趋势分析；实际值会随架构变化
DTYPE_INFO = {
    "FP32": {"bits": 32, "circuit_weight": 1.00},
    "TF32": {"bits": 32, "circuit_weight": 0.55},  # 10-bit mantissa, 仍占 32 bits，但计算电路更接近 FP16
    "FP16": {"bits": 16, "circuit_weight": 0.22},
    "BF16": {"bits": 16, "circuit_weight": 0.20},
}

# 保守估计 HBM 带宽（用于 ridge point 计算）
HBM_BANDWIDTH_GBPS = 128  # GB/s，RTX 5060 Laptop GDDR6X 约 128-256


def run_experiment(workload_fn, workload_name: str, shapes: dict):
    results = []
    for dtype_name, dtype, _ in DTYPE_CONFIGS:
        print("\n--- " + workload_name + " / " + dtype_name + " ---")

        if dtype_name == "FP32":
            torch.backends.cuda.matmul.allow_tf32 = False
            torch.backends.cudnn.allow_tf32 = False
        elif dtype_name == "TF32":
            torch.backends.cuda.matmul.allow_tf32 = True
            torch.backends.cudnn.allow_tf32 = True

        fn, flops = workload_fn(dtype, **shapes)
        profile = profile_workload(workload_name, fn, dtype_name)

        bytes_used = profile["hbm_bytes"]
        elapsed_s = profile["elapsed_ms"] / 1000

        # compute intensities
        ai_std = flops / bytes_used if bytes_used > 0 else 0
        tflops = flops / elapsed_s / 1e12
        peak_tflops = peak_results[dtype_name]["tflops"]
        ridge_point = peak_tflops / HBM_BANDWIDTH_GBPS
        rho = ai_std / ridge_point if ridge_point > 0 else 0

        # bit-level intensity
        w = DTYPE_INFO[dtype_name]["circuit_weight"]
        b = DTYPE_INFO[dtype_name]["bits"]
        i_bit = (flops * w) / (bytes_used * 8) * (32 / b) if bytes_used > 0 else 0
        # 上面乘以 32/b 是为了把分母都统一到 32-bit 等效 bytes

        results.append({
            "dtype": dtype_name,
            "workload": workload_name,
            "elapsed_ms": profile["elapsed_ms"],
            "tflops": tflops,
            "peak_tflops": peak_tflops,
            "flops": flops,
            "bytes": bytes_used,
            "ai_std": ai_std,
            "i_bit": i_bit,
            "ridge_point": ridge_point,
            "rho": rho,
        })

        print(f"  elapsed: {profile['elapsed_ms']:.2f} ms")
        print(f"  achieved: {tflops:.2f} TFLOP/s / peak: {peak_tflops:.2f} TFLOP/s")
        print(f"  HBM bytes: {bytes_to_str(bytes_used)}")
        print(f"  AI_std: {ai_std:.2f} FLOP/byte")
        print(f"  I_bit: {i_bit:.2f}")
        print(f"  ridge point: {ridge_point:.2f}")
        print(f"  rho: {rho:.3f}")

    return results


# GEMM experiment
gemm_results = run_experiment(gemm_workload, "GEMM", {"m": 4096, "k": 4096, "n": 4096})

# MLP experiment
mlp_results = run_experiment(mlp_workload, "MLP", {"dim": 2048, "num_layers": 4, "batch_size": 512})

all_results = gemm_results + mlp_results



## Part 6: 可视化结果

我们画四张图：

1. **Achieved TFLOP/s vs Peak TFLOP/s**：看 utilization。
2. **AI_std 随 dtype 变化**：传统 roofline 指标。
3. **I_bit 随 dtype 变化**：bit-level 强度。
4. **ρ（到 ridge point 的距离）随 dtype 变化**：核心结论图。


In [ ]:

def plot_results(results: List[Dict], title: str):
    dtypes = [r["dtype"] for r in results]
    x = np.arange(len(dtypes))
    width = 0.35

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Plot 1: Achieved vs Peak TFLOP/s
    ax = axes[0, 0]
    achieved = [r["tflops"] for r in results]
    peak = [r["peak_tflops"] for r in results]
    ax.bar(x - width/2, achieved, width, label="Achieved", color="#4a90d9")
    ax.bar(x + width/2, peak, width, label="Peak", color="#90caf9")
    ax.set_xticks(x)
    ax.set_xticklabels(dtypes)
    ax.set_ylabel("TFLOP/s")
    ax.set_title(f"{title}: Achieved vs Peak TFLOP/s")
    ax.legend()
    ax.grid(axis="y", linestyle="--", alpha=0.5)

    # Plot 2: AI_std
    ax = axes[0, 1]
    ai_std = [r["ai_std"] for r in results]
    ax.bar(dtypes, ai_std, color="#81c784")
    ax.set_ylabel("FLOPs / byte")
    ax.set_title(f"{title}: Traditional Arithmetic Intensity (AI_std)")
    ax.grid(axis="y", linestyle="--", alpha=0.5)

    # Plot 3: I_bit
    ax = axes[1, 0]
    i_bit = [r["i_bit"] for r in results]
    ax.bar(dtypes, i_bit, color="#ffb74d")
    ax.set_ylabel("Bit-level intensity")
    ax.set_title(f"{title}: Bit-level Intensity (I_bit)")
    ax.grid(axis="y", linestyle="--", alpha=0.5)

    # Plot 4: rho
    ax = axes[1, 1]
    rho = [r["rho"] for r in results]
    colors = ["#e57373" if v < 1 else "#64b5f6" for v in rho]
    ax.bar(dtypes, rho, color=colors)
    ax.axhline(y=1.0, color="red", linestyle="--", label="Ridge point (rho=1)")
    ax.set_ylabel("rho = AI_std / (P_peak / B_mem)")
    ax.set_title(f"{title}: Distance to Ridge Point (rho)")
    ax.legend()
    ax.grid(axis="y", linestyle="--", alpha=0.5)

    plt.tight_layout()
    plt.show()


plot_results(gemm_results, "GEMM")
plot_results(mlp_results, "MLP")



## Part 7: 传统 Roofline 图

在同一个坐标系里画出各 dtype 的 ridge point 和各 workload 的 $AI_{std}$。

- 横轴：$AI_{std}$ (FLOPs/byte)
- 纵轴：Performance (TFLOP/s)
- 每条 dtype 的 roofline 由 $P_{peak}$ 和 $B_{mem}$ 决定
- 点表示该 workload 在该 dtype 下的实测位置


In [ ]:

def plot_roofline(results: List[Dict], title: str):
    fig, ax = plt.subplots(figsize=(10, 6))

    ai_range = np.logspace(
        np.log10(min(r["ai_std"] for r in results) * 0.5),
        np.log10(max(r["ai_std"] for r in results) * 2),
        100,
    )

    colors = {"FP32": "#1976d2", "TF32": "#388e3c", "FP16": "#f57c00", "BF16": "#7b1fa2"}

    for r in results:
        dtype = r["dtype"]
        peak = r["peak_tflops"]
        ridge = r["ridge_point"]
        roof = np.minimum(peak, HBM_BANDWIDTH_GBPS * ai_range / 1000)  # GB/s * FLOP/byte / 1000 = TFLOP/s
        ax.plot(ai_range, roof, label=f"{dtype} roofline", color=colors.get(dtype, "#333"), alpha=0.6)
        ax.scatter([r["ai_std"]], [r["tflops"]], color=colors.get(dtype, "#333"), s=100, zorder=5)
        ax.annotate(
            dtype,
            (r["ai_std"], r["tflops"]),
            textcoords="offset points",
            xytext=(5, 5),
            fontsize=9,
        )

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Arithmetic Intensity AI_std (FLOPs/byte)")
    ax.set_ylabel("Performance (TFLOP/s)")
    ax.set_title(f"{title}: Roofline Plot")
    ax.legend()
    ax.grid(True, which="both", linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()


plot_roofline(gemm_results, "GEMM")
plot_roofline(mlp_results, "MLP")



## Part 8: 讨论与结论

### 预期发现

1. **GEMM**：大矩阵乘法的 $AI_{std}$ 很高，FP16 的 $\rho$ 可能接近或大于 1，接近 compute-bound。
2. **MLP**：activation/weight 访问更频繁，$AI_{std}$ 较低，FP16 的 $\rho$ 可能反而比 FP32 小，意味着更难喂满 FP16 的 peak。
3. **TF32 的有趣位置**：
   - bytes 和 FP32 一样（32 bits）。
   - 但 compute 接近 FP16。
   - 所以它的 ridge point 可能很高，但 $AI_{std}$ 不变 → $\rho$ 可能显著降低。

### 局限

- HBM bytes 来自 PyTorch allocator 统计，不是精确的 HBM traffic。
- `circuit_weight` 是经验估计，需要更底层的 profiling（ncu、功耗计）校准。
- 没有考虑 Tensor Core 的 layout transform、softmax/reduction 等 memory-bound 算子。

### 下一步

- 用 `ncu` 精确测量 `dram__bytes.sum`。
- 用 `nvidia-smi` 功耗反推 energy-per-FLOP。
- 把实验扩展到 transformer layer（attention 是典型的 memory-bound）。
